# 第74章 交互式数据分析报告

把整理后的业务指标组织成一份交互式分析报告，练习Plotly图表、筛选维度和HTML导出。

## 项目背景

管理者需要一份能够快速回答‘销售趋势如何、哪个区域贡献最大、利润是否同步增长’的轻量报告。报告应保留默认可读状态，同时给读者留下探索空间。

## 学习目标

- 准备报告级宽表
- 制作可悬浮查看的趋势图
- 使用下拉控件切换指标
- 导出可分享的HTML报告


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| month | 月份 | 报告时间维度 |
| region | 区域 | 华东、华南、华北、西南 |
| sales | 销售额 | 单位：万元 |
| profit | 利润 | 单位：万元 |
| orders | 订单数 | 订单数量 |
| conversion | 转化率 | 访问到支付的比例 |

## 数据质量检查清单

- 月份是否按时间排序
- 销售额和利润是否为非负数
- 区域与月份组合是否完整
- 利润率是否处于合理范围
- 图表标题和单位是否一致


## 项目任务

1. 准备报告数据集
2. 检查指标质量并计算利润率
3. 制作销售额与利润趋势
4. 制作区域对比图和指标切换控件
5. 导出HTML并列出报告验收项


## 1. 准备报告数据

报告数据通常是一张已经聚合到展示粒度的宽表，便于图表共享筛选维度。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

months = pd.date_range("2026-01-01", periods=6, freq="MS")
regions = ["华东", "华南", "华北", "西南"]
report = pd.DataFrame({
    "month": np.tile(months, len(regions)),
    "region": np.repeat(regions, len(months)),
})
report["sales"] = np.tile([120, 148, 139, 176, 205, 228], len(regions)) * np.repeat([1.0, 0.82, 0.73, 0.58], len(months))
report["profit"] = report["sales"] * np.tile([0.15, 0.16, 0.14, 0.18, 0.19, 0.21], len(regions))
report["orders"] = (report["sales"] * np.tile([9.8, 10.2, 9.4, 10.1, 10.5, 10.8], len(regions))).round().astype(int)
report["conversion"] = np.clip(report["sales"] / 2200, 0.03, 0.3)
report["month_label"] = report["month"].dt.strftime("%m月")
print(report.head())
print("记录数:", len(report))


## 2. 检查指标并计算利润率

报告中的派生指标应在数据层统一计算，避免不同图表各自使用不同公式。


In [ ]:
print("缺失值:\n", report.isna().sum())
print("负销售额:", (report["sales"] < 0).sum())
print("利润大于销售额:", (report["profit"] > report["sales"]).sum())
print("月份是否有序:", report["month"].is_monotonic_increasing)

report["profit_rate"] = report["profit"] / report["sales"]
regional_total = report.groupby("region", as_index=False).agg(
    sales=("sales", "sum"),
    profit=("profit", "sum"),
    orders=("orders", "sum"),
)
regional_total["profit_rate"] = regional_total["profit"] / regional_total["sales"]
print("区域摘要:\n", regional_total.round(3))


## 3. 制作趋势和区域图

同一份报告可以用子图同时展示趋势和结构；Hover补充精确值，标题保留业务单位。


In [ ]:
monthly_total = report.groupby(["month", "month_label"], as_index=False)[["sales", "profit"]].sum()
fig = make_subplots(rows=1, cols=2, subplot_titles=["月度销售额与利润", "区域销售额"])
fig.add_trace(go.Scatter(x=monthly_total["month_label"], y=monthly_total["sales"], mode="lines+markers", name="销售额", hovertemplate="%{x}<br>销售额：%{y:.1f} 万元<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly_total["month_label"], y=monthly_total["profit"], mode="lines+markers", name="利润", hovertemplate="%{x}<br>利润：%{y:.1f} 万元<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=regional_total["region"], y=regional_total["sales"], name="区域销售额", hovertemplate="%{x}<br>销售额：%{y:.1f} 万元<extra></extra>"), row=1, col=2)
fig.update_layout(title="经营分析报告概览", template="plotly_white", hovermode="x unified")
fig.update_xaxes(title_text="月份", row=1, col=1)
fig.update_yaxes(title_text="金额（万元）", row=1, col=1)
fig.update_xaxes(title_text="区域", row=1, col=2)
fig.update_yaxes(title_text="销售额（万元）", row=1, col=2)
fig.show()


## 4. 增加指标切换和导出

交互控件只解决一个清晰任务：在销售额、利润和订单数之间切换查看。


In [ ]:
metric_config = {
    "销售额": ("sales", "销售额（万元）"),
    "利润": ("profit", "利润（万元）"),
    "订单数": ("orders", "订单数"),
}
fig = go.Figure()
for index, (label, (column, axis_title)) in enumerate(metric_config.items()):
    monthly_metric = report.groupby("month_label", sort=False)[column].sum()
    fig.add_trace(go.Bar(x=monthly_metric.index, y=monthly_metric.values, name=label, visible=index == 0))
    if index == 0:
        default_axis_title = axis_title
buttons = []
for index, (label, (_, axis_title)) in enumerate(metric_config.items()):
    buttons.append({"label": label, "method": "update", "args": [{"visible": [item == index for item in range(len(metric_config))]}, {"title": f"月度{label}", "yaxis": {"title": axis_title}}]})
fig.update_layout(title="月度销售额", template="plotly_white", updatemenus=[{"buttons": buttons, "direction": "down", "x": 1.02, "y": 1.15}], yaxis_title=default_axis_title)
fig.show()

report_path = "/tmp/course_interactive_report.html"
fig.write_html(report_path, include_plotlyjs="cdn")
print("已导出:", report_path)
print("HTML大小（字符）:", len(fig.to_html(include_plotlyjs="cdn")))


## 结论与表达

- 报告默认视图应先回答核心问题
- 交互控件用于探索，不替代静态结论
- 导出前要检查字体、单位、图例和默认状态


## 项目验收清单

- 包含至少两种图表结构
- Hover能看到业务字段
- 下拉控件能切换指标
- HTML文件成功写入临时目录
- 报告默认状态不依赖用户操作

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

把整理后的业务指标组织成一份交互式分析报告，练习Plotly图表、筛选维度和HTML导出。


### 你已经完成

- 准备报告级宽表
- 制作可悬浮查看的趋势图
- 使用下拉控件切换指标
- 导出可分享的HTML报告


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 准备报告数据集 |
| 步骤 2 | 检查指标质量并计算利润率 |
| 步骤 3 | 制作销售额与利润趋势 |
| 步骤 4 | 制作区域对比图和指标切换控件 |
| 步骤 5 | 导出HTML并列出报告验收项 |


### 质量与结论提醒

- 月份是否按时间排序
- 销售额和利润是否为非负数
- 区域与月份组合是否完整
- 报告默认视图应先回答核心问题
- 交互控件用于探索，不替代静态结论
- 导出前要检查字体、单位、图例和默认状态


### 项目交付检查

- [ ] 包含至少两种图表结构
- [ ] Hover能看到业务字段
- [ ] 下拉控件能切换指标
- [ ] HTML文件成功写入临时目录
- [ ] 报告默认状态不依赖用户操作
